# Summarize the 2,352 Hearings in Shards

| Item | Description |
|---|---|
| Goal | Run map-reduce summarization over every hearing in the science committee sample: one row per hearing, one summary each. The run is split ten ways without overlap, and it resumes: a hearing counts as done when its call log holds a reduce line, so a task that hits the wall clock is submitted again, not restarted. |
| Input | `hearings_sst.parquet` from D4. The model Qwen3.5-4B, chosen in T1.1.2, in bfloat16 without quantization. |
| Output | In `Interim/sst_Qwen3.5-4B` of the T1.2 scratch folder: `status/<packageId>.jsonl`, every model call with stage, level, input hash, token counts, time, and full output; `shards/summaries_<shard>.parquet`, one row per hearing the shard finished. T1.2.2 joins the shards. |
| Run | Ran on 2026-08-21 as a ten-task SLURM array, one NVIDIA A40 per task, about 12 hours each at the measured 4.66 seconds per thousand words; the job script is `J - Jobs/run_T1.2.1_summarize_sst.sbatch`. The cell outputs shown are those of shard 0's executed copy. Not rerun: 121 GPU hours, and every later section was built on these summaries. Step 12 reads the logs of all ten shards and was executed on 2026-10-31. |

## Step 1. Import packages

In [ ]:
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings")
NOTEBOOK_NAME = "T1.2.1 - Production - Summarize the Sample in Shards"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
INPUT_FOLDER = SCRATCH_FOLDER / "D - Data Collection and Processing/D4 - SST Sample Construction" / "Processed"
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T1 - Text Summarization" / ".env"

MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)

## Step 3. Read the run settings from the environment

Every setting comes from the job script, so the same notebook serves all ten tasks and the shard it ran is recorded in its output. Shard `i` takes every tenth hearing starting at `i`, on the file's own order. The shards are therefore disjoint, cover the sample exactly, and stay within 4 percent of each other by word count.

In [1]:
MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
SHARD = int(os.environ.get("SHARD", 0))
N_SHARDS = int(os.environ.get("N_SHARDS", 1))
CHUNK_TOKENS = int(os.environ.get("CHUNK_TOKENS", 6144))
RUN_NAME = os.environ.get("RUN_NAME", f"sst_{MODEL.split('/')[-1]}")

RUN_FOLDER = INTERIM_FOLDER / RUN_NAME
STATUS_FOLDER = RUN_FOLDER / "status"
SHARD_FOLDER = RUN_FOLDER / "shards"
STATUS_FOLDER.mkdir(parents=True, exist_ok=True)
SHARD_FOLDER.mkdir(parents=True, exist_ok=True)

# A token is needed only for gated models (the two Llama checkpoints). It is read from .env, never written into the notebook.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")

print(f"run     : {RUN_NAME}")
print(f"model   : {MODEL}")
print(f"shard   : {SHARD} of {N_SHARDS}")
print(f"chunk   : {CHUNK_TOKENS:,} tokens")
print(f"gpu     : {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"output  : {RUN_FOLDER}")
print(f"sampling: {GENERATION}   prompt {PROMPT_VERSION}")

run     : sst_Qwen3.5-4B
model   : Qwen/Qwen3.5-4B
shard   : 0 of 10
chunk   : 6,144 tokens
gpu     : NVIDIA A40, 48 GB
output  : /scratch2/acroh/projects/llm_method/data/interim/summaries/sst_Qwen3.5-4B
sampling: {'do_sample': True, 'temperature': 0.1}   prompt hearing-mapreduce-1.0


## Step 4. Select this shard's hearings

A hearing is finished when its log records a reduce call. The shard reads its own logs at startup and skips what is already there.

In [2]:
sst = pd.read_parquet(INPUT_FOLDER / "hearings_sst.parquet")
work = sst.iloc[SHARD::N_SHARDS].reset_index(drop=True)

done = set()
for path in STATUS_FOLDER.glob("*.jsonl"):
    for line in path.read_text().splitlines():
        if line.strip() and json.loads(line).get("stage") == "reduce":
            done.add(path.stem)
            break

todo = work.loc[~work["packageId"].isin(done)].reset_index(drop=True)

print(f"sample      {len(sst):>6,} hearings, {sst['n_words'].sum() / 1e6:.1f}M words")
print(f"this shard  {len(work):>6,} hearings, {work['n_words'].sum() / 1e6:.1f}M words")
print(f"  already done {len(work) - len(todo):>5,}")
print(f"  to process   {len(todo):>5,}")
if len(todo):
    print(f"\nprojected: {todo['n_words'].sum() / 1000 * 4.66 / 3600:.1f} h at the measured 4.66 s/kword")

sample       2,352 hearings, 93.6M words
this shard     236 hearings, 9.3M words
  already done     0
  to process     236

projected: 12.0 h at the measured 4.66 s/kword


## Step 5. Load the model

In [3]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params, "
      f"{next(model.parameters()).dtype}, {torch.cuda.memory_allocated() / 1e9:.1f} GB on device")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

loaded in 12s — 4.21B params, torch.bfloat16, 8.4 GB on device


## Step 6. Define the three prompts

The same three prompts as T1.1.1, frozen under the version `hearing-mapreduce-1.0` that every logged call carries.

In [ ]:
# One prompt for all three stages fails in a specific way: the reduce step then treats summaries as speech and re-attributes positions.
PROMPT_VERSION = "hearing-mapreduce-1.0"  # stamped on every logged call, so a summary can be traced to the wording that produced it
REDUCE_MAX_TOKENS = 512  # the summaries are later embedded by a 512-token encoder, so a longer summary would be cut, not longer
REDUCE_TARGET_WORDS = "200 to 250"  # the model writes about 1.55 tokens per word, so this lands near 390 tokens with room to finish
GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, kept from the earlier pipeline

MAP_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
              "States congressional hearings. You report only what the transcript states.")

MAP_USER = """Below is one consecutive segment of a congressional hearing transcript. Each speaking turn is labelled with the speaker.

Summarize this segment in about 150 words. Cover:
- the subjects raised, and what was said about them;
- the position each named speaker takes, attributed to that speaker;
- any numbers, dollar amounts, dates, and bill or act names, copied exactly as they appear.

If the segment is procedural or ceremonial and contains none of these, say so in one sentence.

--- SEGMENT ---
{text}
--- END OF SEGMENT ---

Write only the summary of the segment above."""

COLLAPSE_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
                   "States congressional hearings. You condense summaries without adding to them.")

COLLAPSE_USER = """Below are summaries of consecutive segments of one congressional hearing, in the order they occurred. They are already summaries, not transcript.

Condense them into one account of about 200 words that keeps the named speakers, their positions, and the numbers and legislation. This is a middle portion of the hearing, so do not write an overall conclusion about the hearing.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the condensed account."""

REDUCE_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
                 "States congressional hearings. You write from the summaries given and add "
                 "nothing that is not in them.")

REDUCE_USER = """Below are summaries of every segment of one congressional hearing, in order. They are already summaries, not transcript.

Write a single summary of the hearing as a whole, in about {target_words} words. Cover:
- what the hearing was about;
- the main subjects, the most heavily discussed first;
- the position each named speaker took;
- any point where speakers took opposing positions, naming both sides;
- the key numbers and the legislation referred to, exactly as given.

Where the segment summaries do not establish one of these, leave it out rather than inferring it.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the summary of the hearing, in about {target_words} words, as continuous prose."""

## Step 7. Define how speaking turns are packed into chunks

Whole turns, in order, within the budget; only a turn longer than the budget is divided; questions for the record are left out.

In [ ]:
def format_turn(turn):
    """One turn as the model sees it: a speaker label, then what was said. The chair has no surname, so the honorific alone is the label."""
    name = turn.get("surname") or ""
    honorific = turn.get("honorific") or ""
    who = f"{honorific} {name}".strip() or "Unidentified speaker"
    if turn.get("section") == "qfr":
        who = f"{who} (submitted for the record)"
    return f"{who}: {turn['text']}"


def encoded_length(tokenizer, text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def token_slice(text, tokenizer, max_tokens):
    """Cut text into pieces that each re-encode within the budget. decode then encode is not the identity, so every piece is re-checked."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = []
    start = 0
    while start < len(ids):
        take = min(max_tokens, len(ids) - start)
        piece = tokenizer.decode(ids[start:start + take])
        while take > 64 and encoded_length(tokenizer, piece) > max_tokens:
            take = int(take * 0.9)
            piece = tokenizer.decode(ids[start:start + take])
        pieces.append(piece)
        start += take
    return pieces


def split_long_turn(text, tokenizer, max_tokens):
    """Divide one over-long turn at paragraph boundaries, and a paragraph that is still too long by tokens."""
    join_cost = encoded_length(tokenizer, "\n\n")
    out = []
    buffer = []
    buffer_tokens = 0
    for paragraph in text.split("\n\n"):
        n = encoded_length(tokenizer, paragraph)
        if n > max_tokens:
            if buffer:
                out.append("\n\n".join(buffer))
                buffer, buffer_tokens = [], 0
            out.extend(token_slice(paragraph, tokenizer, max_tokens))
            continue
        cost = n + (join_cost if buffer else 0)
        if buffer_tokens + cost > max_tokens:
            out.append("\n\n".join(buffer))
            buffer, buffer_tokens = [], 0
            cost = n
        buffer.append(paragraph)
        buffer_tokens += cost
    if buffer:
        out.append("\n\n".join(buffer))
    # Re-encode every piece: separators cost tokens and the tokenizer re-merges across a cut.
    verified = []
    for piece in out:
        if encoded_length(tokenizer, piece) <= max_tokens:
            verified.append(piece)
        else:
            verified.extend(token_slice(piece, tokenizer, max_tokens))
    return verified


def chunk_turns(turns, tokenizer, max_tokens, spoken_only=True):
    """Group consecutive whole turns into chunks of at most max_tokens tokens. Only a turn longer than the budget is divided."""
    selected = [t for t in turns if not (spoken_only and t.get("section") == "qfr")]
    join_cost = encoded_length(tokenizer, "\n\n")  # the blank lines between turns are tokens too
    chunks = []
    current = []
    current_tokens = 0
    current_speakers = []

    def flush(split=False):
        nonlocal current, current_tokens, current_speakers
        if not current:
            return
        text = "\n\n".join(current)
        speakers = sorted(set(current_speakers))
        if encoded_length(tokenizer, text) > max_tokens:  # the emitted text is what counts, not the sum of its parts
            for piece in split_long_turn(text, tokenizer, max_tokens):
                chunks.append({"text": piece, "n_turns": len(current), "n_tokens": encoded_length(tokenizer, piece),
                               "speakers": speakers, "split_turn": True})
        else:
            chunks.append({"text": text, "n_turns": len(current), "n_tokens": encoded_length(tokenizer, text),
                           "speakers": speakers, "split_turn": split})
        current, current_tokens, current_speakers = [], 0, []

    for turn in selected:
        rendered = format_turn(turn)
        n = len(tokenizer.encode(rendered, add_special_tokens=False))
        if n > max_tokens:  # one turn that cannot fit: emit what is pending, then divide this turn
            flush()
            for piece in split_long_turn(rendered, tokenizer, max_tokens):
                chunks.append({"text": piece, "n_turns": 1, "n_tokens": encoded_length(tokenizer, piece),
                               "speakers": [turn.get("surname") or "Unidentified speaker"], "split_turn": True})
            continue
        cost = n + (join_cost if current else 0)
        if current_tokens + cost > max_tokens:
            flush()
            cost = n
        current.append(rendered)
        current_tokens += cost
        current_speakers.append(turn.get("surname") or "Unidentified speaker")
    flush()
    return chunks

## Step 8. Define one model call

In [ ]:
def render(tokenizer, system, user):
    """Apply the model's chat template, with the thinking mode off where the template offers it."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, system, user, max_new_tokens):
    """One prompt, one completion, fresh context. Also returns whether the completion ran into the token cap."""
    prompt = render(tokenizer, system, user)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = ids["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **GENERATION)
    generated = out[0][n_in:]
    hit_cap = len(generated) >= max_new_tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), hit_cap


SENTENCE_END = tuple('.!?') + tuple(f'{p}{q}' for p in '.!?' for q in '"\')]')


def trim_to_sentence(text):
    """Cut a completion that hit the cap back to its last complete sentence."""
    text = text.rstrip()
    if not text or text.endswith(SENTENCE_END):
        return text
    cut = max(text.rfind(mark) for mark in '.!?')
    if cut <= 0:
        return text
    end = cut + 1
    while end < len(text) and text[end] in '"\')]':
        end += 1
    return text[:end].rstrip()


def summarize_chunk(chunk_text, model, tokenizer, max_new_tokens=320):
    """Map: one transcript segment to a summary of what was said in it."""
    text, hit_cap = generate(model, tokenizer, MAP_SYSTEM, MAP_USER.format(text=chunk_text), max_new_tokens)
    return trim_to_sentence(text) if hit_cap else text


def collapse_summaries(joined, model, tokenizer, max_new_tokens=384):
    """Collapse: a run of segment summaries to one shorter account."""
    text, hit_cap = generate(model, tokenizer, COLLAPSE_SYSTEM, COLLAPSE_USER.format(text=joined), max_new_tokens)
    return trim_to_sentence(text) if hit_cap else text


def reduce_summaries(joined, model, tokenizer, max_new_tokens=REDUCE_MAX_TOKENS):
    """Reduce: all segment summaries to the hearing's single summary, within the cap the downstream encoder needs."""
    text, hit_cap = generate(model, tokenizer, REDUCE_SYSTEM, REDUCE_USER.format(text=joined, target_words=REDUCE_TARGET_WORDS), max_new_tokens)
    return trim_to_sentence(text) if hit_cap else text

## Step 9. Define the map-reduce loop and the call log

In [ ]:
class CallLog:
    """One JSON line per model call, written as it happens: stage, level, position, a hash of the exact input, token counts, time, and the full output."""

    def __init__(self, path, document_id, model_name, prompt_version=None):
        self.path = Path(path) if path else None
        self.document_id = document_id
        self.model_name = model_name
        self.prompt_version = prompt_version or PROMPT_VERSION
        self.records = []
        if self.path:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, stage, level, index, n_inputs, prompt_text, output, tokenizer, seconds):
        record = {
            "document_id": self.document_id, "model": self.model_name, "prompt_version": self.prompt_version,
            "stage": stage, "level": level, "index": index, "n_inputs": n_inputs,
            "input_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
            "input_chars": len(prompt_text),
            "input_tokens": len(tokenizer.encode(prompt_text, add_special_tokens=False)),
            "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
            "output_chars": len(output), "seconds": round(seconds, 2),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "output": output,
        }
        self.records.append(record)
        if self.path:
            with self.path.open("a") as log_file:
                log_file.write(json.dumps(record) + "\n")
                log_file.flush()


def map_reduce(chunks, model, tokenizer, budget_tokens, max_levels=8, log=None, document_id="", model_name=""):
    """Map every chunk, collapse the summaries until they fit in one call, then reduce once."""
    if log is None:
        log = CallLog(None, document_id, model_name)

    level_summaries = []
    for i, chunk in enumerate(chunks):
        t0 = time.time()
        out = summarize_chunk(chunk, model, tokenizer)
        log.write("map", 0, i, 1, MAP_USER.format(text=chunk), out, tokenizer, time.time() - t0)
        level_summaries.append(out)
    levels = [list(level_summaries)]

    level = 0
    for _ in range(max_levels):  # a runaway guard, not a target; the level reached is reported per hearing
        joined = "\n\n".join(level_summaries)
        if len(tokenizer.encode(joined, add_special_tokens=False)) <= budget_tokens:
            break
        level += 1
        groups, current, current_tokens = [], [], 0  # regroup into batches that fit, then condense each
        for summary in level_summaries:
            n = len(tokenizer.encode(summary, add_special_tokens=False))
            if current and current_tokens + n > budget_tokens:
                groups.append(current)
                current, current_tokens = [], 0
            current.append(summary)
            current_tokens += n
        if current:
            groups.append(current)
        collapsed = []
        for i, group in enumerate(groups):
            text = "\n\n".join(group)
            t0 = time.time()
            out = collapse_summaries(text, model, tokenizer)
            log.write("collapse", level, i, len(group), COLLAPSE_USER.format(text=text), out, tokenizer, time.time() - t0)
            collapsed.append(out)
        level_summaries = collapsed
        levels.append(list(level_summaries))

    joined = "\n\n".join(level_summaries)
    t0 = time.time()
    final = reduce_summaries(joined, model, tokenizer)
    log.write("reduce", level + 1, 0, len(level_summaries), REDUCE_USER.format(text=joined, target_words=REDUCE_TARGET_WORDS), final, tokenizer, time.time() - t0)

    return {"final_summary": final, "final_tokens": len(tokenizer.encode(final, add_special_tokens=False)), "levels": levels,
            "n_levels": len(levels), "n_collapse_rounds": level, "n_calls": len(log.records), "n_chunks": len(chunks), "log": log}

## Step 10. Summarize the shard

The shard file is rewritten after every hearing, so a task killed at the wall clock leaves a readable file. The generator is seeded from the `packageId`, so a hearing run again after a crash, or in a differently sized array, draws the same completions. One line is printed per ten hearings.

In [4]:
SHARD_FILE = SHARD_FOLDER / f"summaries_{SHARD:02d}.parquet"
rows = []
if SHARD_FILE.exists():
    rows = pd.read_parquet(SHARD_FILE).to_dict("records")

started = time.time()
for n, hearing in enumerate(todo.itertuples(), 1):
    chunks = chunk_turns(hearing.turns, tokenizer, CHUNK_TOKENS, spoken_only=True)
    log = CallLog(STATUS_FOLDER / f"{hearing.packageId}.jsonl", hearing.packageId, MODEL)
    torch.manual_seed(int.from_bytes(hashlib.sha256(hearing.packageId.encode()).digest()[:4], "big"))
    t0 = time.time()
    try:
        out = map_reduce([c["text"] for c in chunks], model, tokenizer, budget_tokens=CHUNK_TOKENS, log=log)
        status, summary = "ok", out["final_summary"]
    except Exception as exc:  # recorded per hearing; the shard goes on
        status, summary = f"error: {type(exc).__name__}: {str(exc)[:160]}", ""
        out = {"final_tokens": 0, "n_chunks": len(chunks), "n_calls": len(log.records), "n_collapse_rounds": 0}

    rows.append({
        "packageId": hearing.packageId, "model": MODEL, "prompt_version": PROMPT_VERSION, "shard": SHARD,
        "n_words": int(hearing.n_words), "n_turns": int(hearing.n_turns),
        "n_chunks": out["n_chunks"], "n_calls": out["n_calls"], "n_collapse_rounds": out["n_collapse_rounds"],
        "summary_tokens": out["final_tokens"], "seconds": round(time.time() - t0, 1),
        "status": status, "summary": summary,
    })
    pd.DataFrame(rows).to_parquet(SHARD_FILE, compression="zstd", index=False)

    if n % 10 == 0 or n == len(todo):
        rate = sum(r["n_words"] for r in rows[-n:]) / 1000 / (time.time() - started)
        left = (todo["n_words"].iloc[n:].sum() / 1000 / rate / 3600) if rate else 0
        print(f"  {n:>4}/{len(todo)}  {hearing.packageId:<20} {out['n_chunks']:>3}ch {out['n_calls']:>3}calls  {rate:.2f} kword/s  eta {left:.1f} h")

print(f"\nshard {SHARD}: {len(rows):,} hearings in {(time.time() - started) / 3600:.2f} h")
print(pd.DataFrame(rows)["status"].str.split(":").str[0].value_counts().to_string())

    10/236  CHRG-106shrg85657     14ch  15calls  0.25 kword/s  eta 10.0 h


    20/236  CHRG-107shrg87972      6ch   7calls  0.25 kword/s  eta 9.5 h


    30/236  CHRG-108shrg20249     17ch  18calls  0.25 kword/s  eta 8.8 h


    40/236  CHRG-108hhrg92615     16ch  17calls  0.24 kword/s  eta 8.5 h


    50/236  CHRG-109hhrg99574     15ch  16calls  0.24 kword/s  eta 8.1 h


    60/236  CHRG-109shrg27705     13ch  14calls  0.24 kword/s  eta 7.7 h


... (14 outputs left out)

   210/236  CHRG-117hhrg45792      7ch   8calls  0.24 kword/s  eta 1.2 h


   220/236  CHRG-117shrg55075      8ch   9calls  0.24 kword/s  eta 0.8 h


   230/236  CHRG-118shrg61852     28ch  31calls  0.24 kword/s  eta 0.3 h


   236/236  CHRG-119shrg63981     24ch  25calls  0.24 kword/s  eta 0.0 h

shard 0: 236 hearings in 10.66 h
status
ok    236


## Step 11. Check the shard's file

Each task checks only what it wrote. T1.2.2 checks that the sample is complete once all ten have finished.

In [5]:
back = pd.read_parquet(SHARD_FILE)
expected = set(work["packageId"])

assert set(back["packageId"]) <= expected, "this shard wrote a hearing outside its slice"
print(f"1) every row belongs to shard {SHARD} ({len(back):,} of {len(work):,})")

assert not back["packageId"].duplicated().any(), "a hearing was summarized twice"
print("2) no hearing summarized twice")

failed = back.loc[back["status"] != "ok"]
print(f"3) failures: {len(failed)}")
for r in failed.head(5).itertuples():
    print(f"     {r.packageId}: {r.status}")

good = back.loc[back["status"] == "ok"]
over = good.loc[good["summary_tokens"] > REDUCE_MAX_TOKENS]
assert len(over) == 0, f"{len(over)} summaries exceed the {REDUCE_MAX_TOKENS}-token cap"
print(f"4) every summary within {REDUCE_MAX_TOKENS} tokens (median {good['summary_tokens'].median():.0f}, max {good['summary_tokens'].max()})")

empty = good.loc[good["summary"].str.strip().eq("")]
assert len(empty) == 0, f"{len(empty)} rows marked ok carry no summary"
print("5) no row marked ok is empty")

logged = sum(1 for p in STATUS_FOLDER.glob("*.jsonl") if p.stem in set(back["packageId"]))
print(f"6) call logs on disk for {logged:,} of this shard's {len(back):,} hearings")

1) every row belongs to shard 0 (236 of 236)
2) no hearing summarized twice
3) failures: 0
4) every summary within 512 tokens (median 398, max 511)
5) no row marked ok is empty
6) call logs on disk for 236 of this shard's 236 hearings


## Step 12. Read the logs of the completed run

Evidence of the whole run as it happened: the ten shard files and the 2,352 call logs. This cell reads saved files only, so it can run at any time without a GPU.

In [1]:
shard_files = sorted((INTERIM_FOLDER / "sst_Qwen3.5-4B" / "shards").glob("summaries_*.parquet"))
shards = pd.concat([pd.read_parquet(p) for p in shard_files], ignore_index=True)
print(f"{len(shard_files)} shard files, {len(shards):,} hearings, status {shards['status'].value_counts().to_dict()}")
print(f"model {shards['model'].unique().tolist()}, prompt version {shards['prompt_version'].unique().tolist()}")
per_shard = shards.groupby("shard").agg(hearings=("packageId", "size"), words=("n_words", "sum"), gpu_hours=("seconds", "sum"))
per_shard["gpu_hours"] = per_shard["gpu_hours"] / 3600
print(per_shard.to_string())
print(f"\ntotal GPU time {shards['seconds'].sum() / 3600:,.1f} h for {shards['n_words'].sum() / 1e6:.1f}M words "
      f"= {shards['seconds'].sum() / (shards['n_words'].sum() / 1000):.2f} s per thousand words")
print(f"chunks per hearing: median {shards['n_chunks'].median():.0f}, max {shards['n_chunks'].max()}; "
      f"collapse rounds {shards['n_collapse_rounds'].value_counts().sort_index().to_dict()}")
print(f"summary tokens: median {shards['summary_tokens'].median():.0f}, max {shards['summary_tokens'].max()}")

calls = []
for log_file in (INTERIM_FOLDER / "sst_Qwen3.5-4B" / "status").glob("*.jsonl"):
    for line in log_file.read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            calls.append((log_file.stem, r["stage"], r["input_tokens"], r["output_tokens"], r["seconds"], r["at"]))
calls = pd.DataFrame(calls, columns=["packageId", "stage", "input_tokens", "output_tokens", "seconds", "at"])
print(f"\n{len(calls):,} logged model calls for {calls['packageId'].nunique():,} hearings, "
      f"from {calls['at'].min()[:16]} to {calls['at'].max()[:16]} UTC")
print(calls.groupby("stage").agg(calls=("packageId", "size"), median_input_tokens=("input_tokens", "median"),
                                 median_output_tokens=("output_tokens", "median"), median_seconds=("seconds", "median")).to_string())
with_reduce = calls.loc[calls["stage"] == "reduce", "packageId"].nunique()
assert with_reduce == len(shards) == 2352, f"{with_reduce} hearings have a reduce call against {len(shards)} rows"
print(f"\nevery one of the {with_reduce:,} hearings has exactly one reduce call in its log")

10 shard files, 2,352 hearings, status {'ok': 2352}
model ['Qwen/Qwen3.5-4B'], prompt version ['hearing-mapreduce-1.0']
       hearings    words  gpu_hours
shard                              
0           236  9261788  10.638944
1           236  9150366  12.493500
2           235  9663653  12.910417
3           235  9346012  12.436000
4           235  9705102  10.953861
5           235  9780254  10.918500
6           235  9095631   9.988056
7           235  8842044  12.272722
8           235  9526965  10.065056
9           235  9272221  10.049194

total GPU time 112.7 h for 93.6M words = 4.33 s per thousand words
chunks per hearing: median 11, max 122; collapse rounds {0: 2259, 1: 93}
summary tokens: median 396, max 512

29,264 logged model calls for 2,352 hearings, from 2026-08-21T01:55 to 2026-08-21T23:21 UTC
          calls  median_input_tokens  median_output_tokens  median_seconds
stage                                                                     
collapse    190             